# Redrob Intelligent Candidate Ranker — Demo

Runs the full pipeline end-to-end on a **50-candidate sample** (drawn from
`sample_candidates.json` supplied with the challenge).

**Runtime on a free Colab CPU: ~3–4 min** (model download dominates on first run).

Steps:
1. Clone repo & install dependencies
2. Prepare sample data (convert JSON array → JSONL)
3. `precompute.py` — extract features, build BM25 index, encode career embeddings
4. `rank.py` — hybrid retrieval + structural scoring → ranked CSV
5. Display ranked output

> Note: the official validator requires exactly 100 output rows (for the full
> 100 K pool). With this 50-candidate demo pool, candidates failing the yoe ≥ 5
> gate reduce the output to however many pass all gates — typically 25–35 rows.
> The pipeline logic is identical to the production run.

In [ ]:
# ── Step 1: Clone repo ────────────────────────────────────────────────────────
!git clone https://github.com/HARJAPAN2005/india-runs-candidate-ranker.git
%cd india-runs-candidate-ranker

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────────────
# Colab ships with numpy/pandas already; this mainly adds bm25s + sentence-transformers
!pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
# ── Step 3: Prepare sample data ───────────────────────────────────────────────
# sample_candidates.json is a JSON array; precompute.py expects one JSON object
# per line (JSONL). Convert here.
import json

with open('India_runs_data_and_ai_challenge/sample_candidates.json', encoding='utf-8') as f:
    candidates = json.load(f)

print(f'Sample pool: {len(candidates)} candidates')

out_path = 'India_runs_data_and_ai_challenge/candidates.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

print(f'Written to {out_path} ✓')

In [ ]:
# ── Step 4: precompute.py ─────────────────────────────────────────────────────
# Downloads all-MiniLM-L6-v2 on first run (~86 MB, ~2 min).
# Subsequent runs skip the embedding step (cached in artifacts/).
!python precompute.py

In [ ]:
# ── Step 5: rank.py ───────────────────────────────────────────────────────────
# Zero network calls — loads model from artifacts/model/ (TRANSFORMERS_OFFLINE=1).
!python rank.py

In [ ]:
# ── Step 6: Display results ───────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('submission.csv')
print(f'Ranked {len(df)} candidates from {len(candidates)}-candidate demo pool')
print(f'Score range: {df["score"].min():.4f} – {df["score"].max():.4f}')
print()

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.max_rows', 50)
df[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# ── Step 7: Show gate counts & artifact sizes ─────────────────────────────────
import os

feat = pd.read_parquet('artifacts/candidate_features.parquet')
print(f'Total candidates in demo pool:  {len(feat)}')
print(f'Honeypots flagged:              {feat["is_honeypot"].sum()}')
print(f'yoe < 5 (gated out):            {(feat["yoe"] < 5).sum()}')
print(f'Services-only careers:          {feat["services_only_career"].sum()}')
print(f'Ranked output rows:             {len(df)}')
print()

for root, dirs, files in os.walk('artifacts'):
    dirs.sort()
    for fn in sorted(files):
        path = os.path.join(root, fn)
        size = os.path.getsize(path)
        rel  = os.path.relpath(path, 'artifacts')
        print(f'  artifacts/{rel:<45}  {size/1024:.1f} KB')